# V18-v2 Result 4.2 IA vs AR Verification

**Purpose:** Generate data for rewriting Result 4.2 (IA vs AR: Chronic vs Acute Resolution)

**Approach:** For each gene claimed in 4.2, compute:
1. NL→IT (IT context)
2. NL→IA
3. NL→AR
4. IA vs AR (direct comparison)
5. IT→IA (transition context from 4.1)

**Target genes:** GNLY, FOXP3, LAYN, TIGIT, DNMT1 + additional IA-AR discriminators

**Prerequisite:** adata loaded in memory (from previous notebook)

In [ ]:
# Cell 1: Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import os, time
import warnings
warnings.filterwarnings('ignore')

try:
    _ = adata.shape
    print(f'adata already loaded: {adata.shape}')
except:
    from google.colab import drive
    drive.mount('/content/drive')
    import scanpy as sc
    DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
    print('Loading h5ad...')
    adata = sc.read_h5ad(DATA_PATH)
    print(f'Loaded: {adata.shape}')

RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2'
SAVE_DIR = os.path.join(RESULTS_DIR, 'Result4-2-supplementary_26Mar12')
os.makedirs(SAVE_DIR, exist_ok=True)

obs = adata.obs.copy()

# Column detection (same as 4.1 notebook)
COL_STAGE = 'Stage' if 'Stage' in obs.columns else [c for c in obs.columns if 'stage' in c.lower()][0]
COL_LINEAGE = 'major_lineage' if 'major_lineage' in obs.columns else [c for c in obs.columns if 'lineage' in c.lower()][0]
if 'tissue' in obs.columns:
    COL_TISSUE = 'tissue'
elif 'Tissue' in obs.columns:
    COL_TISSUE = 'Tissue'
else:
    for c in ['sample', 'Sample', 'orig.ident']:
        if c in obs.columns:
            obs['tissue_derived'] = obs[c].apply(
                lambda x: 'Liver' if ('Liver' in str(x) or '_L_' in str(x) or str(x).endswith('_L'))
                else ('Blood' if ('PBMC' in str(x) or '_P_' in str(x) or str(x).endswith('_P') or 'Blood' in str(x))
                else 'Unknown'))
            COL_TISSUE = 'tissue_derived'
            break
COL_DONOR = None
for c in ['donor', 'Donor', 'patient', 'subject', 'donor_id']:
    if c in obs.columns:
        COL_DONOR = c
        break
if COL_DONOR is None:
    for c in ['sample', 'Sample', 'orig.ident']:
        if c in obs.columns:
            obs['donor_derived'] = obs[c].astype(str).str.split('_').str[1]
            COL_DONOR = 'donor_derived'
            break

print(f'Stage: {COL_STAGE} → {sorted(obs[COL_STAGE].unique())}')
print(f'Lineage: {COL_LINEAGE}')
print(f'Tissue: {COL_TISSUE}')
print(f'Donor: {COL_DONOR}')

In [ ]:
# Cell 2: Pre-extract gene expressions (optimized)
t0 = time.time()

# 4.2 target genes + broader IA-AR discriminators
target_genes = [
    # Current 4.2 claims
    'GNLY', 'FOXP3', 'LAYN', 'TIGIT', 'DNMT1',
    # Additional IA-AR candidates
    'CTLA4', 'TOX', 'GZMB', 'PRF1', 'GZMK',
    'DNMT3A', 'TET2', 'TGFB1', 'LGALS9',
    'SOCS1', 'SOCS3', 'PRDM1', 'BCL6',
    'JAK1', 'MTOR', 'AIM2', 'MEFV',
    'TYROBP', 'FCER1G',  # PlasmaB cytotoxic
]
target_genes = [g for g in target_genes if g in adata.var_names]
print(f'Genes to test: {len(target_genes)}')

gene_expr = {}
for gi, gene in enumerate(target_genes):
    try:
        col = adata[:, gene].X
        if hasattr(col, 'toarray'):
            col = col.toarray().flatten()
        else:
            col = np.asarray(col).flatten()
        gene_expr[gene] = col
        if (gi + 1) % 10 == 0:
            print(f'  {gi+1}/{len(target_genes)} extracted...')
    except Exception as e:
        print(f'  ⚠️ {gene}: {e}')

print(f'Pre-extraction done in {time.time()-t0:.1f}s')

In [ ]:
# Cell 3: Fast donor-level test function (pre-extracted)

def fast_donor_test(gene, lineage, tissue, group_a, group_b):
    if gene not in gene_expr:
        return None
    mask = (
        (obs[COL_STAGE].isin([group_a, group_b])) &
        (obs[COL_LINEAGE] == lineage) &
        (obs[COL_TISSUE] == tissue)
    )
    if mask.sum() == 0:
        return None
    cell_indices = np.where(mask.values)[0]
    expr = gene_expr[gene][cell_indices]
    temp = obs.loc[mask, [COL_DONOR, COL_STAGE]].copy()
    temp['expr'] = expr
    donor_means = temp.groupby([COL_DONOR, COL_STAGE])['expr'].mean().reset_index()
    vals_a = donor_means[donor_means[COL_STAGE] == group_a]['expr'].dropna().values
    vals_b = donor_means[donor_means[COL_STAGE] == group_b]['expr'].dropna().values
    n_a, n_b = len(vals_a), len(vals_b)
    if n_a < 2 or n_b < 2:
        return None
    mean_a, mean_b = float(np.mean(vals_a)), float(np.mean(vals_b))
    try:
        _, p_val = mannwhitneyu(vals_a, vals_b, alternative='two-sided')
        p_val = float(p_val)
    except:
        p_val = 1.0
    if mean_a > 1e-10:
        pct = (mean_b - mean_a) / mean_a * 100
    else:
        pct = float('inf') if mean_b > 1e-10 else 0.0
    n_consistent = sum(1 for va in vals_a for vb in vals_b
                       if (mean_b > mean_a and vb > va) or (mean_b <= mean_a and vb < va))
    return {
        'p': round(p_val, 4),
        'pct': round(pct, 1),
        'dir': '↑' if mean_b > mean_a else '↓',
        'sig': '★' if p_val < 0.05 else ('†' if p_val < 0.10 else ''),
        'cons': f'{n_consistent}/{n_a * n_b}',
        'mean_a': round(mean_a, 4), 'mean_b': round(mean_b, 4),
        'n_a': n_a, 'n_b': n_b,
    }

def fmt(res):
    if res is None:
        return 'ND'
    return f'{res["sig"]}{res["dir"]}{abs(res["pct"]):.0f}% p={res["p"]:.3f}'

print('Functions defined ✅')

In [ ]:
# Cell 4: PRIORITY — 4.2 target genes across 5 comparisons
# NL→IT, NL→IA, NL→AR, IA vs AR, IT→IA

priority_genes = ['GNLY', 'FOXP3', 'LAYN', 'TIGIT', 'DNMT1']
lineages = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']
tissues = ['Liver', 'Blood']
comparisons = [
    ('NL', 'IT'),
    ('NL', 'IA'),
    ('NL', 'AR'),
    ('IA', 'AR'),   # IA vs AR direct
    ('IT', 'IA'),   # transition context
]

print('='*130)
print('RESULT 4.2 PRIORITY GENES: Full Disease Spectrum Comparison')
print('='*130)
print(f'{"Gene":>8s} | {"Lineage":>8s} | {"Tissue":>6s} | '
      f'{"NL→IT":>18s} | {"NL→IA":>18s} | {"NL→AR":>18s} | '
      f'{"IA vs AR":>18s} | {"IT→IA":>18s}')
print('-'*130)

priority_results = []

for gene in priority_genes:
    if gene not in gene_expr:
        print(f'{gene:>8s} | NOT IN DATASET')
        continue
    for lin in lineages:
        for tis in tissues:
            results = {}
            any_sig = False
            for ga, gb in comparisons:
                res = fast_donor_test(gene, lin, tis, ga, gb)
                results[f'{ga}→{gb}'] = res
                if res and res['p'] < 0.05:
                    any_sig = True
            
            if any_sig:
                row = {
                    'Gene': gene, 'Lineage': lin, 'Tissue': tis,
                }
                for comp_name, res in results.items():
                    row[f'{comp_name}_change'] = fmt(res)
                    row[f'{comp_name}_p'] = res['p'] if res else None
                priority_results.append(row)
                
                print(f'{gene:>8s} | {lin:>8s} | {tis:>6s} | '
                      f'{fmt(results.get("NL→IT")):>18s} | '
                      f'{fmt(results.get("NL→IA")):>18s} | '
                      f'{fmt(results.get("NL→AR")):>18s} | '
                      f'{fmt(results.get("IA→AR")):>18s} | '
                      f'{fmt(results.get("IT→IA")):>18s}')

print(f'\nTotal significant rows: {len(priority_results)}')

In [ ]:
# Cell 5: BROAD SCAN — All target genes, IA vs AR comparison
# Focus on IA vs AR direct comparison to find discriminators

print('='*100)
print('IA vs AR DIRECT COMPARISON — All Target Genes (Significant Only)')
print('='*100)
print(f'{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | '
      f'{"IA mean":>8s} | {"AR mean":>8s} | {"Dir":>3s} | '
      f'{"% Change":>10s} | {"p":>8s} | {"Cons":>8s} | {"Who wins?":>10s}')
print('-'*100)

ia_ar_results = []

for gene in target_genes:
    for lin in lineages:
        for tis in tissues:
            res = fast_donor_test(gene, lin, tis, 'IA', 'AR')
            if res and res['p'] < 0.10:  # include trends
                who = 'AR↑' if res['mean_b'] > res['mean_a'] else 'IA↑'
                ia_ar_results.append({
                    'gene': gene, 'lineage': lin, 'tissue': tis,
                    'ia_mean': res['mean_a'], 'ar_mean': res['mean_b'],
                    'direction': res['dir'], 'pct': res['pct'],
                    'p': res['p'], 'sig': res['sig'],
                    'consistency': res['cons'], 'who_wins': who,
                })
                print(f'{res["sig"]:>1s}{gene:>9s} | {lin:>8s} | {tis:>6s} | '
                      f'{res["mean_a"]:>8.4f} | {res["mean_b"]:>8.4f} | '
                      f'{res["dir"]:>3s} | {res["pct"]:>9.1f}% | '
                      f'{res["p"]:>8.4f} | {res["cons"]:>8s} | {who:>10s}')

print(f'\nTotal IA vs AR significant/trend: {len(ia_ar_results)}')

In [ ]:
# Cell 6: IT PERSPECTIVE — For each IA vs AR discriminator,
# show the IT-phase status (NL→IT) to contextualize

print('='*140)
print('IA vs AR DISCRIMINATORS WITH IT CONTEXT')
print('(For each IA-AR difference, how was the gene in IT?)')
print('='*140)
print(f'{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | '
      f'{"NL→IT":>18s} | {"NL→IA":>18s} | {"NL→AR":>18s} | '
      f'{"IA vs AR":>18s} | {"Interpretation":>25s}')
print('-'*140)

for r in sorted(ia_ar_results, key=lambda x: x['p']):
    gene, lin, tis = r['gene'], r['lineage'], r['tissue']
    
    nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
    nl_ia = fast_donor_test(gene, lin, tis, 'NL', 'IA')
    nl_ar = fast_donor_test(gene, lin, tis, 'NL', 'AR')
    ia_ar = fast_donor_test(gene, lin, tis, 'IA', 'AR')
    
    # Interpretation
    it_sig = nl_it and nl_it['p'] < 0.05
    ia_sig = nl_ia and nl_ia['p'] < 0.05
    ar_sig = nl_ar and nl_ar['p'] < 0.05
    
    if it_sig and ia_sig and not ar_sig:
        interp = 'IT+IA persist, AR normal'
    elif it_sig and not ia_sig and ar_sig:
        interp = 'IT+AR change, IA reverts'
    elif it_sig and ia_sig and ar_sig:
        interp = 'All changed vs NL'
    elif not it_sig and ia_sig and not ar_sig:
        interp = 'IA-specific'
    elif it_sig and not ia_sig and not ar_sig:
        interp = 'IT-specific'
    elif not it_sig and not ia_sig and ar_sig:
        interp = 'AR-specific'
    else:
        interp = 'Complex'
    
    print(f'{r["sig"]:>1s}{gene:>9s} | {lin:>8s} | {tis:>6s} | '
          f'{fmt(nl_it):>18s} | {fmt(nl_ia):>18s} | '
          f'{fmt(nl_ar):>18s} | {fmt(ia_ar):>18s} | {interp:>25s}')

In [ ]:
# Cell 7: Detailed donor-level values for key 4.2 genes

def get_donor_expression_fast(gene, lineage, tissue):
    """Get donor-level expression across all stages using pre-extracted data."""
    if gene not in gene_expr:
        return pd.DataFrame()
    mask = (obs[COL_LINEAGE] == lineage) & (obs[COL_TISSUE] == tissue)
    cells = obs[mask].copy()
    cell_indices = np.where(mask.values)[0]
    cells['expr'] = gene_expr[gene][cell_indices]
    return cells.groupby([COL_DONOR, COL_STAGE])['expr'].mean().reset_index()

STAGE_ORDER = ['NL', 'IT', 'IA', 'AR', 'CR']

key_combos = [
    ('GNLY', 'CD8_T', 'Blood'),
    ('GNLY', 'CD8_T', 'Liver'),
    ('GNLY', 'PlasmaB', 'Liver'),
    ('FOXP3', 'CD4_T', 'Liver'),
    ('FOXP3', 'CD4_T', 'Blood'),
    ('FOXP3', 'CD8_T', 'Liver'),
    ('LAYN', 'CD4_T', 'Liver'),
    ('LAYN', 'NK', 'Liver'),
    ('TIGIT', 'CD4_T', 'Liver'),
    ('TIGIT', 'CD8_T', 'Liver'),
    ('DNMT1', 'PlasmaB', 'Liver'),
    ('DNMT1', 'PlasmaB', 'Blood'),
    ('DNMT1', 'Myeloid', 'Liver'),
    ('DNMT1', 'Myeloid', 'Blood'),
]

for gene, lin, tis in key_combos:
    df = get_donor_expression_fast(gene, lin, tis)
    if len(df) == 0:
        continue
    
    means = {}
    for stage in STAGE_ORDER:
        vals = df[df[COL_STAGE] == stage]['expr'].dropna().values
        if len(vals) > 0:
            means[stage] = np.mean(vals)
    
    if len(means) < 3:
        continue
    
    # IA vs AR comparison
    ia_ar = fast_donor_test(gene, lin, tis, 'IA', 'AR')
    ia_ar_str = fmt(ia_ar) if ia_ar else 'ND'
    
    vals_str = ' | '.join(f'{s}={means.get(s, 0):.4f}' for s in STAGE_ORDER if s in means)
    print(f'{gene:>8s} {lin:>8s} {tis:>6s}: {vals_str} | IA-AR: {ia_ar_str}')

In [ ]:
# Cell 8: Save comprehensive IA vs AR supplementary table

supp_rows = []
comparisons_all = [('NL','IT'), ('NL','IA'), ('NL','AR'), ('IA','AR'), ('IT','IA')]

for gene in target_genes:
    for lin in lineages:
        for tis in tissues:
            row = {'Gene': gene, 'Lineage': lin, 'Tissue': tis}
            any_sig = False
            for ga, gb in comparisons_all:
                prefix = f'{ga}→{gb}'
                res = fast_donor_test(gene, lin, tis, ga, gb)
                if res:
                    row[f'{prefix}_change'] = f'{res["dir"]}{abs(res["pct"]):.1f}%'
                    row[f'{prefix}_p'] = res['p']
                    row[f'{prefix}_sig'] = res['sig']
                    row[f'{prefix}_cons'] = res['cons']
                    if res['p'] < 0.05:
                        any_sig = True
                else:
                    row[f'{prefix}_change'] = 'ND'
                    row[f'{prefix}_p'] = None
                    row[f'{prefix}_sig'] = ''
                    row[f'{prefix}_cons'] = ''
            if any_sig:
                supp_rows.append(row)

df_supp = pd.DataFrame(supp_rows)
supp_path = os.path.join(SAVE_DIR, 'SuppTable_IA_AR_Comprehensive.csv')
df_supp.to_csv(supp_path, index=False)
print(f'Saved: {supp_path}')
print(f'Rows: {len(df_supp)}')

# Also save IA vs AR significant only
if ia_ar_results:
    df_iaar = pd.DataFrame(ia_ar_results)
    iaar_path = os.path.join(SAVE_DIR, 'IA_vs_AR_significant.csv')
    df_iaar.to_csv(iaar_path, index=False)
    print(f'Saved: {iaar_path}')

In [ ]:
# Cell 9: Summary for manuscript revision

print('='*70)
print('MANUSCRIPT REVISION GUIDE — Result 4.2')
print('='*70)
print()
print('KEY QUESTION: How do IT-phase changes relate to IA vs AR differences?')
print()
print('For each gene, the narrative should be:')
print('  1. What happened in IT (NL→IT)?')
print('  2. Does it persist in IA (NL→IA)?')
print('  3. Is AR different from IA (IA vs AR)?')
print('  4. What does this tell us about IT→IA→CR vs AR resolution?')
print()
print('EXPECTED NARRATIVE STRUCTURE:')
print('  "In the IT phase, [gene] was [up/down] in [lineage].')
print('   This change [persisted/resolved] in IA, whereas AR showed')
print('   [different/similar] levels, suggesting that [interpretation]."')
print()
print('✅ All data generated for Result 4.2 revision.')